# Eksperimen, Preprocessing & Feature Engineering
Notebook ini didedikasikan untuk memvisualisasikan seluruh proses di balik layar secara *step-by-step*, sesuai dengan panduan proyek:
- Distribusi Data setelah Split
- Hasil Data Augmentasi
- Proses Preprocessing (Resizing, Grayscale, CLAHE, Blur, Thresholding/Morfologi)
- Ekstraksi Fitur (Canny, DWT, Flatten, Scaling)

In [ ]:
import sys
import os
# Jika script diupload sebagai dataset Kaggle, hilangkan tanda pagar di bawah ini dan sesuaikan foldernya
sys.path.append('/kaggle/input/datasets/emageeeee/pcd-k23')

import cv2
import pywt
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from pathlib import Path
from tensorflow.keras.preprocessing.image import ImageDataGenerator, img_to_array, load_img
from config import CLASSES, IMG_SIZE_CNN
from src.preprocess import download_data_from_kaggle

kaggle_dir = download_data_from_kaggle()
print("Data raw terhubung di:", kaggle_dir)

## 1. Splitting Dataset & Distribusi Data

In [ ]:
def show_split_distribution():
    splits = ['Train', 'Validation', 'Test']
    counts = []
    for split in splits:
        for cls in CLASSES:
            path = kaggle_dir / split / cls
            n_images = len(list(path.glob("*.png")) + list(path.glob("*.jpg")))
            counts.append({'Split': split, 'Class': cls, 'Total': n_images})
            
    df = pd.DataFrame(counts)
    display(df.pivot(index='Split', columns='Class', values='Total'))
    
    plt.figure(figsize=(8,4))
    sns.barplot(data=df, x='Split', y='Total', hue='Class')
    plt.title("Distribusi Data Setelah Splitting")
    plt.show()

show_split_distribution()

## 2. Visualisasi Data Augmentasi

In [ ]:
def show_augmentation():
    # Definisi Augmentasi sesuai dengan notebook awal
    datagen = ImageDataGenerator(
        rescale=1./255, 
        rotation_range=10, 
        width_shift_range=0.2, 
        height_shift_range=0.2,
        zoom_range=0.25, 
        horizontal_flip=True, 
        fill_mode='nearest'
    )
    
    # Ambil 1 contoh gambar
    sample_path = list((kaggle_dir / 'Train' / CLASSES[0]).glob("*.png"))[0]
    img = load_img(sample_path)
    x = img_to_array(img)
    x = x.reshape((1,) + x.shape)
    
    fig, axes = plt.subplots(1, 5, figsize=(20, 4))
    axes[0].imshow(img)
    axes[0].set_title("Original Image")
    axes[0].axis('off')
    
    # Generate 4 gambar augmented
    i = 1
    for batch in datagen.flow(x, batch_size=1):
        axes[i].imshow(batch[0])
        axes[i].set_title(f"Augmented {i}")
        axes[i].axis('off')
        i += 1
        if i > 4:
            break
    plt.show()

show_augmentation()

## 3. Preprocessing (Step-by-Step)

In [ ]:
sample_path = list((kaggle_dir / 'Train' / CLASSES[0]).glob("*.png"))[0]
img_orig = cv2.imread(str(sample_path))
img_rgb = cv2.cvtColor(img_orig, cv2.COLOR_BGR2RGB)

def plot_comparison(img1, img2, title1, title2, cmap1=None, cmap2=None):
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].imshow(img1, cmap=cmap1)
    axes[0].set_title(title1)
    axes[0].axis('off')
    
    axes[1].imshow(img2, cmap=cmap2)
    axes[1].set_title(title2)
    axes[1].axis('off')
    plt.show()

### 3.1 Resizing (Transformasi Linear)

In [ ]:
img_resized = cv2.resize(img_rgb, IMG_SIZE_CNN)
plot_comparison(img_rgb, img_resized, f"Original: {img_rgb.shape}", f"Resized: {img_resized.shape}")

### 3.2 Grayscale Conversion

In [ ]:
img_gray = cv2.cvtColor(img_resized, cv2.COLOR_RGB2GRAY)
plot_comparison(img_resized, img_gray, "RGB", "Grayscale", cmap2='gray')

### 3.3 Histogram Equalization (CLAHE)

In [ ]:
clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
img_clahe = clahe.apply(img_gray)
plot_comparison(img_gray, img_clahe, "Grayscale Biasa", "Setelah CLAHE", cmap1='gray', cmap2='gray')

### 3.4 Noise Filtering (Gaussian Blur)

In [ ]:
img_blur = cv2.GaussianBlur(img_clahe, (5, 5), 0)
plot_comparison(img_clahe, img_blur, "Sebelum Blur", "Sesudah Gaussian Blur", cmap1='gray', cmap2='gray')

### 3.5 Morfologi Citra (Thresholding & Morphological Opening/Closing)
*(Eksperimen Tambahan sesuai catatan notes.txt)*

In [ ]:
_, img_thresh = cv2.threshold(img_blur, 127, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
kernel = np.ones((3,3), np.uint8)
img_opening = cv2.morphologyEx(img_thresh, cv2.MORPH_OPEN, kernel)
img_closing = cv2.morphologyEx(img_opening, cv2.MORPH_CLOSE, kernel)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(img_thresh, cmap='gray')
axes[0].set_title("Thresholding (Otsu)")
axes[0].axis('off')

axes[1].imshow(img_opening, cmap='gray')
axes[1].set_title("Morphological Opening")
axes[1].axis('off')

axes[2].imshow(img_closing, cmap='gray')
axes[2].set_title("Morphological Closing")
axes[2].axis('off')
plt.show()

## 4. Ekstraksi Fitur (ML Klasik)

### 4.1 Edge Detection (Canny)

In [ ]:
img_canny = cv2.Canny(img_blur, 100, 200)
plot_comparison(img_blur, img_canny, "Gambar Preprocessed (Blur)", "Fitur Canny Edge", cmap1='gray', cmap2='gray')

### 4.2 Transformasi Wavelet (DWT)

In [ ]:
coeffs = pywt.dwt2(img_blur, 'haar')
LL, (LH, HL, HH) = coeffs

fig, axes = plt.subplots(2, 2, figsize=(10, 10))
axes[0, 0].imshow(LL, cmap='gray')
axes[0, 0].set_title("LL (Approximation)")
axes[0, 0].axis('off')

axes[0, 1].imshow(LH, cmap='gray')
axes[0, 1].set_title("LH (Horizontal Detail)")
axes[0, 1].axis('off')

axes[1, 0].imshow(HL, cmap='gray')
axes[1, 0].set_title("HL (Vertical Detail)")
axes[1, 0].axis('off')

axes[1, 1].imshow(HH, cmap='gray')
axes[1, 1].set_title("HH (Diagonal Detail)")
axes[1, 1].axis('off')
plt.show()

### 4.3 Flattening & Feature Scaling
Tahap akhir di mana gambar 2D diubah menjadi deretan angka 1D dan dinormalisasi untuk SVM.

In [ ]:
from sklearn.preprocessing import StandardScaler

flatten_canny = img_canny.flatten()
flatten_dwt = LL.flatten()

print("Shape asli Canny:", img_canny.shape)
print("Setelah Flatten (1D):", flatten_canny.shape)

print("\nShape asli DWT (LL):", LL.shape)
print("Setelah Flatten (1D):", flatten_dwt.shape)

# Simulasi StandardScaler (harus 2D input)
scaler = StandardScaler()
scaled_canny = scaler.fit_transform(flatten_canny.reshape(-1, 1))

print("\n5 Nilai Array Flatten (Sebelum Scaling):\n", flatten_canny[:5])
print("\n5 Nilai Array Flatten (Sesudah Scaling / Normalisasi):\n", scaled_canny[:5].ravel())